In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
sys.path.append('..')
warnings.filterwarnings('ignore')

# Import your classes
from src.data import Data
from src.features import Features
from src.visualization import Visualization
from src.correlation import Correlation

# Initialize classes
data_loader = Data(fs=700)
feature_extractor = Features(fs=700)
viz = Visualization()
corr = Correlation()

## 1. Load dataset

In [2]:
ecgs,labels = data_loader.read_dataset("../data/WESAD")


## 2. Cut dataset to chuncks

In [3]:
"""
Create three chunked datasets: 30s, 120s, 300s with majority-vote labels.
"""

chunk_durations = [30, 120, 300]  # seconds
threshold = 0.9

# Store results for each duration
datasets = {}

KEEP_LABELS = {1, 2, 3, 4}

for duration in chunk_durations:
    all_chunks = []
    all_labels = []
    total_discarded = 0
    total_discarded_labels = []
    total_discarded_purities = []
    
    for subj_idx, (ecg, label) in enumerate(zip(ecgs, labels)):
        keep_ecg, keep_labels, discarded = data_loader.get_majority_label_chunks(
            ecg, label, time_in_sec=duration, threshold=threshold
        )
        # Only keep chunks with labels 1,2,3,4 (drop e.g. label 0 / transitions)
        for chunk, lbl in zip(keep_ecg, keep_labels):
            if int(lbl) in KEEP_LABELS:
                all_chunks.append(chunk)
                all_labels.append(lbl)
            else:
                total_discarded += 1
                total_discarded_labels.append(lbl)
                total_discarded_purities.append(1.0)  # pure but unwanted label
        total_discarded += discarded['count']
        total_discarded_labels.extend(discarded['labels'])
        total_discarded_purities.extend(discarded['purities'])
    
    datasets[duration] = {
        'chunks': all_chunks,
        'labels': all_labels,
        'discarded_count': total_discarded,
        'discarded_labels': total_discarded_labels,
        'discarded_purities': total_discarded_purities
    }
    
    print(f"{'='*60}")
    print(f"Chunk duration: {duration} sec")
    print(f"{'='*60}")
    print(f"  Total chunks kept:     {len(all_chunks)}")
    print(f"  Total chunks ignored:  {total_discarded}")
    unique_lbl, counts_lbl = np.unique(all_labels, return_counts=True)
    print(f"  Label distribution (kept):")
    for lbl, cnt in zip(unique_lbl, counts_lbl):
        print(f"    Label {lbl}: {cnt} chunks ({cnt/len(all_labels)*100:.1f}%)")
    if total_discarded > 0:
        print(f"  Mean purity of ignored chunks: {np.mean(total_discarded_purities):.3f}")
    print()

# Quick sanity check
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for duration in chunk_durations:
    d = datasets[duration]
    print(f"  {duration:3d}s: {len(d['chunks']):4d} chunks kept, {d['discarded_count']:3d} ignored")
print("="*60)

Chunk duration: 30 sec
  Total chunks kept:     189
  Total chunks ignored:  187
  Label distribution (kept):
    Label 1: 76 chunks (40.2%)
    Label 2: 42 chunks (22.2%)
    Label 3: 23 chunks (12.2%)
    Label 4: 48 chunks (25.4%)
  Mean purity of ignored chunks: 0.961

Chunk duration: 120 sec
  Total chunks kept:     43
  Total chunks ignored:  50
  Label distribution (kept):
    Label 1: 18 chunks (41.9%)
    Label 2: 10 chunks (23.3%)
    Label 3: 4 chunks (9.3%)
    Label 4: 11 chunks (25.6%)
  Mean purity of ignored chunks: 0.872

Chunk duration: 300 sec
  Total chunks kept:     13
  Total chunks ignored:  24
  Label distribution (kept):
    Label 1: 6 chunks (46.2%)
    Label 2: 2 chunks (15.4%)
    Label 3: 2 chunks (15.4%)
    Label 4: 3 chunks (23.1%)
  Mean purity of ignored chunks: 0.779


SUMMARY
   30s:  189 chunks kept, 187 ignored
  120s:   43 chunks kept,  50 ignored
  300s:   13 chunks kept,  24 ignored


## 3. Get features from the chunked dataset

In [4]:
"""
Extract HRV features for all chunks (30s, 120s, 300s).
"""

feature_dfs = {}  # duration -> DataFrame with features + labels

for duration in chunk_durations:
    print(f"Extracting features for {duration}s chunks...")
    chunks = datasets[duration]['chunks']
    labels = datasets[duration]['labels']
    
    # Use the batch method from Features class
    feats = feature_extractor.get_hrv_features_batch(chunks, verbose=True)
    
    # Convert to DataFrame with labels
    df = feature_extractor.get_dataframe(feats, labels=labels)
    feature_dfs[duration] = df
    
    print(f"  -> {len(df)} samples, {len(df.columns)-1} features + label\n")

# Print column names once
print("Feature columns:", [c for c in feature_dfs[30].columns if c != 'label'])

Extracting features for 30s chunks...
Processing ECG 1/189...
Processing ECG 11/189...
Processing ECG 21/189...
Processing ECG 31/189...
Processing ECG 41/189...
Processing ECG 51/189...
Processing ECG 61/189...
Processing ECG 71/189...
Processing ECG 81/189...
Processing ECG 91/189...
Processing ECG 101/189...
Processing ECG 111/189...
Processing ECG 121/189...
Processing ECG 131/189...
Processing ECG 141/189...
Processing ECG 151/189...
Processing ECG 161/189...
Processing ECG 171/189...
Processing ECG 181/189...
  -> 189 samples, 8 features + label

Extracting features for 120s chunks...
Processing ECG 1/43...
Processing ECG 11/43...
Processing ECG 21/43...
Processing ECG 31/43...
Processing ECG 41/43...
  -> 43 samples, 8 features + label

Extracting features for 300s chunks...
Processing ECG 1/13...
Processing ECG 11/13...
  -> 13 samples, 8 features + label

Feature columns: ['mean_rr', 'mean_hr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf_ratio']


## 4. Make the tables of features the same size


In [5]:
"""
Align feature tables: each individual 30s chunk is a row, paired with its
corresponding longer-duration chunk (120s or 300s) that covers the same
time span. The longer-duration chunk value is repeated for each 30s row.

Mapping:
  - 10 consecutive 30s chunks (300s total) → 300s chunk at same position
     -> yields 10 rows, each with (30s_feat, 300s_feat, label)
  -  4 consecutive 30s chunks (120s total) → 120s chunk at same position
     -> yields 4 rows, each with (30s_feat, 120s_feat, label)
"""

# Filter out frequency-domain features (LF, HF, ratio) which need longer signals
feature_cols = [c for c in feature_dfs[30].columns if c != 'label']
feature_cols = [c for c in feature_cols if 'lf' not in c.lower() and 'hf' not in c.lower()]
print(f"Using features (LF/HF excluded): {feature_cols}")
RATIO_300 = 10  # 300s / 30s
RATIO_120 = 4   # 120s / 30s

# Lists to store expanded rows
rows_30v300 = []
rows_30v120 = []

for lbl in KEEP_LABELS:
    df30 = feature_dfs[30]
    df300 = feature_dfs[300]
    df120 = feature_dfs[120]
    
    idx30 = np.where(df30['label'] == lbl)[0]
    idx300 = np.where(df300['label'] == lbl)[0]
    idx120 = np.where(df120['label'] == lbl)[0]
    
    # --- 30s ↔ 300s (10 : 1) ---
    n_pairs_300 = min(len(idx30) // RATIO_300, len(idx300))
    for p in range(n_pairs_300):
        start_30 = p * RATIO_300
        end_30 = start_30 + RATIO_300
        feats_300 = df300.iloc[idx300[p]][feature_cols].values
        
        for j in range(start_30, end_30):
            feats_30 = df30.iloc[idx30[j]][feature_cols].values
            rows_30v300.append(np.concatenate([feats_30, feats_300, [lbl]]))
    
    # --- 30s ↔ 120s (4 : 1) ---
    n_pairs_120 = min(len(idx30) // RATIO_120, len(idx120))
    for p in range(n_pairs_120):
        start_30 = p * RATIO_120
        end_30 = start_30 + RATIO_120
        feats_120 = df120.iloc[idx120[p]][feature_cols].values
        
        for j in range(start_30, end_30):
            feats_30 = df30.iloc[idx30[j]][feature_cols].values
            rows_30v120.append(np.concatenate([feats_30, feats_120, [lbl]]))

# Build DataFrames: each row = (30s_feat_1..N, 120s_or_300s_feat_1..N, label)
cols_30 = [f'30s_{c}' for c in feature_cols]
cols_300 = [f'300s_{c}' for c in feature_cols]
cols_120 = [f'120s_{c}' for c in feature_cols]

df_30v300 = pd.DataFrame(rows_30v300, columns=cols_30 + cols_300 + ['label'])
df_30v120 = pd.DataFrame(rows_30v120, columns=cols_30 + cols_120 + ['label'])

print(f"df_30v300: {len(df_30v300)} rows (10 rows per 300s chunk)")
print(f"df_30v120: {len(df_30v120)} rows (4 rows per 120s chunk)")
print()
print("First 12 rows of df_30v300 (first 300s block = 10 x 30s rows):")
display(df_30v300.head(12).round(2))
print()
print("First 8 rows of df_30v120 (first 120s block = 4 x 30s rows):")
display(df_30v120.head(8).round(2))

Using features (LF/HF excluded): ['mean_rr', 'mean_hr', 'sdnn', 'rmssd', 'pnn50']
df_30v300: 130 rows (10 rows per 300s chunk)
df_30v120: 172 rows (4 rows per 120s chunk)

First 12 rows of df_30v300 (first 300s block = 10 x 30s rows):


,30s_mean_rr,30s_mean_hr,30s_sdnn,30s_rmssd,30s_pnn50,300s_mean_rr,300s_mean_hr,300s_sdnn,300s_rmssd,300s_pnn50,label
0,829.96,72.76,66.19,58.63,51.52,780.16,77.24,51.37,38.09,18.32,1.0
1,855.58,70.58,69.56,57.86,43.75,780.16,77.24,51.37,38.09,18.32,1.0
2,841.51,71.63,58.46,51.10,30.30,780.16,77.24,51.37,38.09,18.32,1.0
3,770.81,78.33,62.25,43.20,22.22,780.16,77.24,51.37,38.09,18.32,1.0
4,810.86,74.20,43.05,44.82,29.41,780.16,77.24,51.37,38.09,18.32,1.0
5,826.69,73.14,71.61,58.17,29.41,780.16,77.24,51.37,38.09,18.32,1.0
6,807.50,74.76,65.39,54.64,31.43,780.16,77.24,51.37,38.09,18.32,1.0
7,737.14,81.94,60.36,31.60,15.38,780.16,77.24,51.37,38.09,18.32,1.0
8,768.35,78.29,40.24,28.34,5.41,780.16,77.24,51.37,38.09,18.32,1.0
9,773.05,77.88,47.13,37.09,13.51,780.16,77.24,51.37,38.09,18.32,1.0



First 8 rows of df_30v120 (first 120s block = 4 x 30s rows):


,30s_mean_rr,30s_mean_hr,30s_sdnn,30s_rmssd,30s_pnn50,120s_mean_rr,120s_mean_hr,120s_sdnn,120s_rmssd,120s_pnn50,label
0,829.96,72.76,66.19,58.63,51.52,821.83,73.52,69.05,51.56,32.17,1.0
1,855.58,70.58,69.56,57.86,43.75,821.83,73.52,69.05,51.56,32.17,1.0
2,841.51,71.63,58.46,51.10,30.30,821.83,73.52,69.05,51.56,32.17,1.0
3,770.81,78.33,62.25,43.20,22.22,821.83,73.52,69.05,51.56,32.17,1.0
4,810.86,74.20,43.05,44.82,29.41,782.85,77.23,68.61,44.58,19.87,1.0
5,826.69,73.14,71.61,58.17,29.41,782.85,77.23,68.61,44.58,19.87,1.0
6,807.50,74.76,65.39,54.64,31.43,782.85,77.23,68.61,44.58,19.87,1.0
7,737.14,81.94,60.36,31.60,15.38,782.85,77.23,68.61,44.58,19.87,1.0


## 5. Calculate correlation parameters (r,ICC,MAE)

In [6]:
"""
Calculate Pearson r, ICC(2,1), and MAE for each feature
using the Correlation class from src/correlation.py.
"""

import pingouin as pg

results_list = []  # list of dicts, one per feature

for comp_name, df_aligned in [('30s_vs_300s', df_30v300), ('30s_vs_120s', df_30v120)]:
    prefix_short = '30s_'
    prefix_long = '300s_' if comp_name == '30s_vs_300s' else '120s_'
    
    for feat in feature_cols:
        col_short = f'{prefix_short}{feat}'
        col_long = f'{prefix_long}{feat}'
        
        vals_short = df_aligned[col_short].values
        vals_long = df_aligned[col_long].values
        
        # Drop NaN pairs
        mask = ~(np.isnan(vals_short) | np.isnan(vals_long))
        v1 = vals_short[mask]
        v2 = vals_long[mask]
        n_valid = len(v1)
        
        if n_valid < 3:
            r_val, icc_val, mae_val = np.nan, np.nan, np.nan
        else:
            r_val, _ = corr.get_r(v1, v2, method='pearson')
            icc_val, icc_ci = corr.get_icc(v1, v2, icc_type='ICC2')
            mae_val = corr.get_mae(v1, v2)
            
            # Debug: print ICC pingouin table for first feature in each comparison
            if feat == feature_cols[0]:
                n = len(v1)
                test_data = pd.DataFrame({
                    'target': list(range(n)) * 2,
                    'rater': ['method1'] * n + ['method2'] * n,
                    'rating': np.concatenate([v1, v2])
                })
                icc_table = pg.intraclass_corr(data=test_data, targets='target', raters='rater', ratings='rating')
                print(f"\n[DEBUG] {comp_name} / {feat}: pingouin ICC table:")
                display(icc_table.round(4))
                print(f"  -> Extracted ICC2 = {icc_val:.4f}, CI = {icc_ci}")
        
        results_list.append({
            'comparison': comp_name,
            'feature': feat,
            'n_pairs': n_valid,
            'r': r_val,
            'ICC': icc_val,
            'MAE': mae_val
        })

df_results = pd.DataFrame(results_list)

# Display results
for comp in ['30s_vs_300s', '30s_vs_120s']:
    print(f"\n{'='*60}")
    print(f"{comp}")
    print('='*60)
    sub = df_results[df_results['comparison'] == comp][['feature', 'n_pairs', 'r', 'ICC', 'MAE']]
    display(sub.round(4))


[DEBUG] 30s_vs_300s / mean_rr: pingouin ICC table:


,Type,ICC,F,df1,df2,pval,CI95
0,"ICC(1,1)",0.6663,4.9928,129,130,0.0,"[0.56, 0.75]"
1,"ICC(A,1)",0.6707,5.4285,129,129,0.0,"[0.55, 0.76]"
2,"ICC(C,1)",0.6889,5.4285,129,129,0.0,"[0.59, 0.77]"
3,"ICC(1,k)",0.7997,4.9928,129,130,0.0,"[0.72, 0.86]"
4,"ICC(A,k)",0.8029,5.4285,129,129,0.0,"[0.71, 0.86]"
5,"ICC(C,k)",0.8158,5.4285,129,129,0.0,"[0.74, 0.87]"


  -> Extracted ICC2 = 0.6707, CI = (nan, nan)

[DEBUG] 30s_vs_120s / mean_rr: pingouin ICC table:


,Type,ICC,F,df1,df2,pval,CI95
0,"ICC(1,1)",0.9284,26.9191,171,172,0.0,"[0.9, 0.95]"
1,"ICC(A,1)",0.9284,27.2607,171,171,0.0,"[0.9, 0.95]"
2,"ICC(C,1)",0.9292,27.2607,171,171,0.0,"[0.91, 0.95]"
3,"ICC(1,k)",0.9629,26.9191,171,172,0.0,"[0.95, 0.97]"
4,"ICC(A,k)",0.9629,27.2607,171,171,0.0,"[0.95, 0.97]"
5,"ICC(C,k)",0.9633,27.2607,171,171,0.0,"[0.95, 0.97]"


  -> Extracted ICC2 = 0.9284, CI = (nan, nan)

30s_vs_300s


,feature,n_pairs,r,ICC,MAE
0,mean_rr,130,0.7138,0.6707,54.3566
1,mean_hr,130,0.7054,0.6544,6.8965
2,sdnn,130,0.5902,0.5167,20.7205
3,rmssd,130,0.5915,0.5409,14.0757
4,pnn50,130,0.6775,0.6367,9.0455



30s_vs_120s


,feature,n_pairs,r,ICC,MAE
5,mean_rr,172,0.9298,0.9284,29.6657
6,mean_hr,172,0.9439,0.9426,3.2333
7,sdnn,172,0.6855,0.6681,15.2992
8,rmssd,172,0.7457,0.7334,11.3725
9,pnn50,172,0.7757,0.7625,7.4094


## 6. Plot Tables to Show the results

In [7]:
"""
Plot correlation results with green/yellow/red color coding.
- r / ICC: higher is better → green >= 0.8, amber >= 0.6, red < 0.6
- MAE:    lower is better → green <= 0.5, amber <= 1.0, red > 1.0
"""

def color_higher(val):
    if val >= 0.8: return 'background-color: #2d7a3e; color: white; font-weight: bold'
    elif val >= 0.6: return 'background-color: #f4a460; color: black; font-weight: bold'
    else: return 'background-color: #c41e3a; color: white; font-weight: bold'

def color_lower(val):
    if val <= 0.5: return 'background-color: #2d7a3e; color: white; font-weight: bold'
    elif val <= 1.0: return 'background-color: #f4a460; color: black; font-weight: bold'
    else: return 'background-color: #c41e3a; color: white; font-weight: bold'

for comp in ['30s_vs_300s', '30s_vs_120s']:
    sub = df_results[df_results['comparison'] == comp].copy()
    
    # Separate r+ICC (higher=better) and MAE (lower=better)
    r_icc = sub[['feature', 'r', 'ICC']].set_index('feature')
    mae = sub[['feature', 'MAE']].set_index('feature')
    
    styled_r_icc = (r_icc.style
                    .map(color_higher)
                    .format('{:.4f}')
                    .set_caption(f'r and ICC — {comp} (green = better)'))
    
    styled_mae = (mae.style
                  .map(color_lower)
                  .format('{:.4f}')
                  .set_caption(f'MAE — {comp} (green = better = lower)'))
    
    print(f"\n{'='*60}")
    print(f"Metrics — {comp}")
    print('='*60)
    display(styled_r_icc)
    display(styled_mae)


Metrics — 30s_vs_300s


,r,ICC
feature,,
mean_rr,0.7138,0.6707
mean_hr,0.7054,0.6544
sdnn,0.5902,0.5167
rmssd,0.5915,0.5409
pnn50,0.6775,0.6367


,MAE
feature,
mean_rr,54.3566
mean_hr,6.8965
sdnn,20.7205
rmssd,14.0757
pnn50,9.0455



Metrics — 30s_vs_120s


,r,ICC
feature,,
mean_rr,0.9298,0.9284
mean_hr,0.9439,0.9426
sdnn,0.6855,0.6681
rmssd,0.7457,0.7334
pnn50,0.7757,0.7625


,MAE
feature,
mean_rr,29.6657
mean_hr,3.2333
sdnn,15.2992
rmssd,11.3725
pnn50,7.4094


In [8]:
"""
Save the coloured tables as PNG images using the Visualization class.
"""
import os
os.makedirs('../results', exist_ok=True)

for comp in ['30s_vs_300s', '30s_vs_120s']:
    sub = df_results[df_results['comparison'] == comp].copy()
    
    # r + ICC (higher is better)
    r_icc = sub[['feature', 'r', 'ICC']].set_index('feature')
    viz.save_styled_df_as_png(
        r_icc, f'../results/r_icc_{comp}.png',
        higher_is_better_cols=['r', 'ICC']
    )
    
    # MAE (lower is better)
    mae = sub[['feature', 'MAE']].set_index('feature')
    viz.save_styled_df_as_png(
        mae, f'../results/mae_{comp}.png',
        lower_is_better_cols=['MAE']
    )

# --- Combined pivot summary as CSV for backup ---
pivot = df_results.pivot_table(
    index='feature',
    columns='comparison',
    values=['r', 'ICC', 'MAE'],
    aggfunc='first'
)
pivot.columns = [f'{metric}_{comp}' for metric, comp in pivot.columns]
pivot.round(4).to_csv('../results/correlation_summary.csv')
print("Saved correlation_summary.csv")

print("\nAll PNG tables saved to ../results/")
!dir ..\results\*.png /b

Saved styled table to: ../results/r_icc_30s_vs_300s.png
Saved styled table to: ../results/mae_30s_vs_300s.png
Saved styled table to: ../results/r_icc_30s_vs_120s.png
Saved styled table to: ../results/mae_30s_vs_120s.png
Saved correlation_summary.csv

All PNG tables saved to ../results/
mae_30s_vs_120s.png
mae_30s_vs_300s.png
r_icc_30s_vs_120s.png
r_icc_30s_vs_300s.png
